In [ ]:
# Shared data loading and plot styling.
import json
import re
from collections import Counter
from datetime import datetime
from itertools import cycle
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

FONT_FAMILY = 'DejaVu Sans'
PALETTE = [
    '#6FA9A1', '#8ABAD3', '#A6D8D0', '#B7C7E5', '#C8D6B9',
    '#D8B4A6', '#E4C7B7', '#C9C1E8', '#A9C7B8', '#D8E2C8',
    '#E6D6B8', '#BFC8D6', '#D7E6E1', '#E8D8D0',
]
COLORS = {
    'text': '#22313A',
    'grid': '#D8E2E1',
    'spine': '#AFC1C4',
    'panel': '#FBFDFD',
    'background': '#F6FAFA',
    'ink': '#314B52',
    'teal': '#5E9E97',
    'seafoam': '#A7D8D0',
    'sky': '#AFC7E8',
    'coral': '#D9B6A8',
    'sand': '#E7D2B5',
    'lavender': '#C9C1E8',
    'sage': '#C8D6B9',
    'moss': '#A9C7B8',
}

plt.rcParams.update({
    'font.family': FONT_FAMILY,
    'font.size': 11,
    'axes.titlesize': 18,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10.5,
    'legend.title_fontsize': 11,
    'axes.facecolor': COLORS['panel'],
    'figure.facecolor': COLORS['background'],
    'axes.edgecolor': COLORS['spine'],
    'axes.labelcolor': COLORS['text'],
    'xtick.color': COLORS['text'],
    'ytick.color': COLORS['text'],
    'text.color': COLORS['text'],
    'grid.color': COLORS['grid'],
    'grid.alpha': 0.75,
    'axes.prop_cycle': plt.cycler(color=PALETTE),
})

SOUND_DIR = Path('Sound_recordings')
FILLER_DIR = Path('Filler_analysis')
SPEED_DIR = Path('Speed')
PITCH_DIR = Path('Pitch')

for folder in (SOUND_DIR, FILLER_DIR, SPEED_DIR, PITCH_DIR):
    if not folder.exists():
        raise FileNotFoundError(f'Could not find folder: {folder}')


def parse_timestamp(text):
    match = re.search(r'(\d{8}_\d{6})', str(text))
    return match.group(1) if match else None


def parse_datetime(value):
    if value is None:
        return None
    if isinstance(value, datetime):
        return value
    if not isinstance(value, str):
        return None

    candidate = value.replace('Z', '+00:00')
    try:
        parsed = datetime.fromisoformat(candidate)
        return parsed.replace(tzinfo=None)
    except ValueError:
        timestamp = parse_timestamp(value)
        if timestamp:
            try:
                return datetime.strptime(timestamp, '%Y%m%d_%H%M%S')
            except ValueError:
                return None
    return None


def clean_recording_label(recording_id):
    return re.sub(r'_\d{8}_\d{6}$', '', str(recording_id))


def load_json_if_exists(path):
    if path and path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return {}


def resolve_json(folder, recording_id, timestamp=None, prefixes=()):
    candidates = [folder / f'{recording_id}.json']
    if timestamp:
        candidates.append(folder / f'{recording_id}_{timestamp}.json')
        for prefix in prefixes:
            candidates.append(folder / f'{prefix}_{timestamp}.json')
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def recording_sort_key(wav_path):
    recording_id = wav_path.stem
    timestamp = parse_timestamp(recording_id)
    if timestamp:
        try:
            return (0, datetime.strptime(timestamp, '%Y%m%d_%H%M%S'), recording_id.lower())
        except ValueError:
            pass
    return (1, datetime.fromtimestamp(wav_path.stat().st_mtime), recording_id.lower())


def extract_pitch_series(payload):
    if isinstance(payload.get('pitch_series'), list):
        return payload['pitch_series']
    if isinstance(payload.get('pitch_track'), list):
        return payload['pitch_track']
    return []


wav_files = sorted(SOUND_DIR.glob('*.wav'), key=recording_sort_key)
if not wav_files:
    raise FileNotFoundError(f'No .wav files found in {SOUND_DIR}')

records = []
all_filler_counts = Counter()

for wav_path in wav_files:
    recording_id = wav_path.stem
    timestamp = parse_timestamp(recording_id)

    speed_json = load_json_if_exists(resolve_json(SPEED_DIR, recording_id, timestamp, prefixes=('speed',)))
    filler_json = load_json_if_exists(resolve_json(FILLER_DIR, recording_id, timestamp, prefixes=('filler_analysis',)))
    pitch_json = load_json_if_exists(resolve_json(PITCH_DIR, recording_id, timestamp))

    recording_dt = (
        parse_datetime(speed_json.get('created_at'))
        or parse_datetime(filler_json.get('created_at'))
        or parse_datetime(pitch_json.get('created_at'))
    )
    if recording_dt is None and timestamp:
        try:
            recording_dt = datetime.strptime(timestamp, '%Y%m%d_%H%M%S')
        except ValueError:
            recording_dt = None
    if recording_dt is None:
        recording_dt = datetime.fromtimestamp(wav_path.stat().st_mtime)

    filler_counts_raw = filler_json.get('filler_word_counts', {}) if isinstance(filler_json, dict) else {}
    filler_counts = {
        str(word): int(count)
        for word, count in filler_counts_raw.items()
        if isinstance(word, str) and int(count) > 0
    }

    total_words = int(speed_json.get('total_words', filler_json.get('total_words', 0)) or 0)
    total_words_denom = max(total_words, 1)
    total_filler_words = int(filler_json.get('total_filler_words', sum(filler_counts.values())))
    filler_percentage = float(
        filler_json.get(
            'filler_percentage',
            100.0 * total_filler_words / total_words_denom,
        )
    )
    wpm = float(speed_json.get('wpm', 0.0))

    pitch_block = pitch_json.get('pitch', pitch_json)
    pitch_series = extract_pitch_series(pitch_json)
    voiced_semitone_values = np.array(
        [
            abs(float(point['pitch_semitones']))
            for point in pitch_series
            if point.get('voiced') and point.get('pitch_semitones') is not None
        ],
        dtype=float,
    )
    if voiced_semitone_values.size:
        pitch_avg_variation = float(np.mean(voiced_semitone_values))
        pitch_min_variation = float(np.min(voiced_semitone_values))
        pitch_max_variation = float(np.max(voiced_semitone_values))
    else:
        pitch_avg_variation = np.nan
        pitch_min_variation = np.nan
        pitch_max_variation = np.nan

    all_filler_counts.update(filler_counts)

    records.append({
        'recording_id': recording_id,
        'recording_label': clean_recording_label(recording_id),
        'recording_dt': recording_dt,
        'speed_json': speed_json,
        'filler_json': filler_json,
        'pitch_json': pitch_json,
        'total_words': total_words,
        'wpm': wpm,
        'total_filler_words': total_filler_words,
        'filler_percentage': filler_percentage,
        'filler_counts': filler_counts,
        'pitch_summary': pitch_block,
        'pitch_series': pitch_series,
        'pitch_avg_variation': pitch_avg_variation,
        'pitch_min_variation': pitch_min_variation,
        'pitch_max_variation': pitch_max_variation,
    })

filler_words = [word for word, _ in all_filler_counts.most_common()]
if not filler_words:
    filler_words = ['um', 'uh', 'like', 'you know']

word_colors = {
    word: color
    for word, color in zip(filler_words, cycle(PALETTE))
}

records = sorted(records, key=lambda record: (record['recording_dt'], record['recording_id']))
record_labels = [record['recording_label'] for record in records]
recording_times = [record['recording_dt'] for record in records]

print(f'Loaded {len(records)} recordings')
print(f'Filler vocabulary: {filler_words}')
print(f'Date range: {recording_times[0].strftime('%Y-%m-%d %H:%M:%S')} -> {recording_times[-1].strftime('%Y-%m-%d %H:%M:%S')}')
print(f'Total filler words found: {sum(all_filler_counts.values())}')


def style_modern_axes(ax, title, xlabel, ylabel):
    ax.set_title(title, pad=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, which='major', axis='both', linewidth=0.8)
    ax.set_facecolor(COLORS['panel'])
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color(COLORS['spine'])
        ax.spines[spine].set_linewidth(1.0)
    ax.tick_params(axis='both', labelsize=11)


def format_date_axis(ax, fmt='%b %d'):
    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
    ax.figure.autofmt_xdate(rotation=25, ha='right')


: 

In [ ]:
# Plot 1: Filler words as percentages per recording.
if not records:
    print('No recordings available to plot.')
else:
    ordered_records = sorted(records, key=lambda record: (record['filler_percentage'], record['recording_dt']), reverse=True)
    labels = [f"{record['recording_label']}  •  {record['recording_dt'].strftime('%b %d, %Y')}" for record in ordered_records]
    total_percentages = np.array([record['filler_percentage'] for record in ordered_records], dtype=float)
    segment_matrix = np.array(
        [
            [
                100.0 * record['filler_counts'].get(word, 0) / max(record['total_words'], 1)
                for record in ordered_records
            ]
            for word in filler_words
        ],
        dtype=float,
    )

    fig_height = max(6.0, 0.42 * len(ordered_records) + 2.2)
    fig, ax = plt.subplots(figsize=(14, fig_height))
    y_positions = np.arange(len(ordered_records))
    left = np.zeros(len(ordered_records), dtype=float)

    for word, values in zip(filler_words, segment_matrix):
        ax.barh(
            y_positions,
            values,
            left=left,
            color=word_colors[word],
            edgecolor='white',
            linewidth=0.9,
            alpha=0.94,
            label=word,
        )
        left += values

    for y_pos, total_pct in zip(y_positions, total_percentages):
        ax.text(
            float(total_pct) + 0.5,
            float(y_pos),
            f'{total_pct:.1f}%',
            va='center',
            ha='left',
            fontsize=10,
            color=COLORS['ink'],
        )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    style_modern_axes(
        ax,
        'Filler words as percentages per recording',
        'Percentage of all spoken words',
        'Recording',
    )
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
    ax.set_xlim(0, max(5.0, float(np.nanmax(total_percentages)) * 1.22 if total_percentages.size else 5.0))
    ax.legend(ncol=2, frameon=False, loc='upper right', bbox_to_anchor=(1.0, 1.02))
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot 2: Filler words over time in percentages.
if not records:
    print('No recordings available to plot.')
else:
    time_ordered_records = sorted(records, key=lambda record: (record['recording_dt'], record['recording_id']))
    dates = [record['recording_dt'] for record in time_ordered_records]
    total_percentages = np.array([record['filler_percentage'] for record in time_ordered_records], dtype=float)
    percent_matrix = np.array(
        [
            [
                100.0 * record['filler_counts'].get(word, 0) / max(record['total_words'], 1)
                for record in time_ordered_records
            ]
            for word in filler_words
        ],
        dtype=float,
    )

    fig, ax = plt.subplots(figsize=(14, 6.6))
    ax.stackplot(
        dates,
        percent_matrix,
        colors=[word_colors[word] for word in filler_words],
        alpha=0.86,
        labels=filler_words,
        linewidth=0.6,
    )
    ax.plot(
        dates,
        total_percentages,
        color=COLORS['ink'],
        linewidth=2.4,
        label='Total filler %',
    )
    ax.fill_between(dates, total_percentages, color=COLORS['ink'], alpha=0.06)

    style_modern_axes(
        ax,
        'Filler words over time',
        'Recording date',
        'Percentage of spoken words',
    )
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
    format_date_axis(ax, fmt='%b %d')
    ax.set_ylim(0, max(5.0, float(np.nanmax(total_percentages)) * 1.2 if total_percentages.size else 5.0))
    ax.legend(
        ncol=3,
        frameon=False,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.18),
        columnspacing=1.4,
        handlelength=1.8,
        title='Filler word',
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot 3: Pitch range per session over time.
if not records:
    print('No recordings available to plot.')
else:
    pitch_records = [record for record in records if np.isfinite(record['pitch_avg_variation'])]
    if not pitch_records:
        print('No voiced pitch values available to plot.')
    else:
        dates = [record['recording_dt'] for record in pitch_records]
        avg_variation = np.array([record['pitch_avg_variation'] for record in pitch_records], dtype=float)
        min_variation = np.array([record['pitch_min_variation'] for record in pitch_records], dtype=float)
        max_variation = np.array([record['pitch_max_variation'] for record in pitch_records], dtype=float)
        overall_average = float(np.nanmean(avg_variation))
        target_low, target_high = 3.0, 5.0

        fig, ax = plt.subplots(figsize=(14, 6.4))
        ax.axhspan(
            target_low,
            target_high,
            color=COLORS['sage'],
            alpha=0.20,
            label='Target zone',
        )
        ax.fill_between(
            dates,
            min_variation,
            max_variation,
            color=COLORS['seafoam'],
            alpha=0.38,
            label='Min-max band',
        )
        ax.plot(
            dates,
            avg_variation,
            color=COLORS['teal'],
            marker='o',
            markersize=6,
            linewidth=2.5,
            label='Average variation',
        )
        ax.plot(
            dates,
            np.full_like(avg_variation, overall_average),
            color=COLORS['ink'],
            linestyle='--',
            linewidth=1.8,
            label=f'Overall average ({overall_average:.2f} st)',
        )

        style_modern_axes(
            ax,
            'Pitch range per session over time',
            'Session date',
            'Pitch variation (semitones from session median)',
        )
        format_date_axis(ax, fmt='%b %d')
        upper_limit = max(target_high + 2.0, float(np.nanmax(max_variation)) * 1.15 if max_variation.size else target_high + 2.0)
        ax.set_ylim(0, upper_limit)
        ax.legend(frameon=False, loc='upper left')
        plt.tight_layout()
        plt.show()

In [ ]:
# Plot 4: WPM over time with average, target zone, and trend line.
if not records:
    print('No recordings available to plot.')
else:
    wpm_records = [record for record in records if np.isfinite(record['wpm'])]
    if not wpm_records:
        print('No WPM values available to plot.')
    else:
        dates = [record['recording_dt'] for record in wpm_records]
        wpm_values = np.array([record['wpm'] for record in wpm_records], dtype=float)
        avg_wpm = float(np.mean(wpm_values))
        target_low, target_high = 130.0, 150.0

        date_numbers = mdates.date2num(dates)
        if len(wpm_values) >= 2:
            trend_coeffs = np.polyfit(date_numbers, wpm_values, 1)
            trend_line = np.polyval(trend_coeffs, date_numbers)
        else:
            trend_line = wpm_values.copy()

        fig, ax = plt.subplots(figsize=(14, 6.0))
        ax.axhspan(
            target_low,
            target_high,
            color=COLORS['sky'],
            alpha=0.20,
            label='Target zone',
        )
        ax.plot(
            dates,
            wpm_values,
            color=COLORS['teal'],
            marker='o',
            markersize=6,
            linewidth=2.4,
            label='WPM',
        )
        ax.plot(
            dates,
            trend_line,
            color=COLORS['coral'],
            linestyle='--',
            linewidth=2.3,
            label='Trend line',
        )
        ax.axhline(
            avg_wpm,
            color=COLORS['ink'],
            linestyle=':',
            linewidth=2.0,
            label=f'Average WPM ({avg_wpm:.1f})',
        )

        style_modern_axes(
            ax,
            'Words per minute over time',
            'Session date',
            'Words per minute (WPM)',
        )
        format_date_axis(ax, fmt='%b %d')
        ax.set_ylim(0, max(180.0, float(np.nanmax(wpm_values)) * 1.18 if wpm_values.size else 180.0))
        ax.legend(frameon=False, loc='upper left')
        plt.tight_layout()
        plt.show()